# Carvana quarterly scenarios: assumptions and missing evidence

**Question: what evidence and assumptions would we need to turn observed vehicle activity into a quarterly unit scenario?** This notebook explains the arithmetic and its limits. It does not produce a validated Carvana sales forecast.

Read the **settings and quarter coverage**, then the **observed activity** if you select retained cycles. The analyst-input table is deliberately empty. Finish with the synthetic revision example and the evidence checklist below. Notebook 23 is the specialist proxy reference; Notebook 24 reviews the selected status experiment.

## Settings and retained inputs

The Ordinary settings cell below contains the main choices. Changing them selects local evidence; it does not collect or import anything.

| Setting | Default / what you can change |
|---|---|
| `AS_OF` | Historical cutoff `2026-09-08T13:00:00Z`; use a timezone-aware timestamp to review another evidence date. |
| `QUARTER` | `2026Q3`; keep the cutoff within or after the quarter you select. |
| `TIMEZONE` | `America/New_York`, matching the daily collection dates. |
| `CYCLE_REPORTS` | Empty list: no daily cycles are selected automatically. Supply existing cycle-report paths to inspect retained observations. |
| `DATABASE` | Existing daily-history SQLite file, opened read-only by the reader. Keep it paired with its cycle reports. |

For one retained example, select [the September 11 cycle](../data/experiments/carvana_daily/2026-09-11/cycle.json) and use cutoff `2026-09-11T16:41:25.820875+00:00`, with the existing daily database. Inspect that date in `calendar` and its row in `daily_activity`. Selecting one date leaves the other quarter dates missing; it does not provide a complete quarter. The default remains an empty selection, so **NO DAILY EVIDENCE is an expected result**.

`research_status` shows the quarter, observed population, completed/missing dates and evidence age. `estimated_retail_units` remains missing even when selected inventory dates are complete. A missing result is not zero sales. Each inventory sweep covers an interval, not continuous observation.

The target definition used here includes Marketplace partner retail units and is net of returns. Website inventory also includes reserved/purchase-in-progress and some unfinished vehicles. See the existing [Q2 2026 filing citation, Key Operating Metrics](https://www.sec.gov/Archives/edgar/data/1690820/000169082026000055/cvna-20260630.htm) and [results publication](https://investors.carvana.com/news-releases/2026/07-29-2026-210525685). A publication date alone does not identify intraday availability; no company benchmark is loaded automatically.
**Learning:** **00 -> 10 -> 11 -> 20 -> 24 -> 30**. For routine review start with
20 and 24; open this notebook when evaluating quarterly assumptions or saved forecasts.
The default is a historical teaching replay, not today's company estimate.

| Main output | Meaning of empty / unavailable |
| --- | --- |
| `calendar`, `research_status`, `daily_activity` | Selected observed dates and gaps; no cycles means no daily evidence. |
| `analyst_scenario` | An explicit assumption scenario, or empty when no inputs were selected / eligibility is blocked. |
| `bridge` | Synthetic 63-to-93 unit arithmetic only; never included as a real forecast. |
| `forecast_vintages`, `accuracy_rows` | All selected vintages and strictly prospective scored rows; no forecasts means accuracy is unvalidated. |

Try the retained one-date example above and inspect its missing quarter dates.
Then, in the synthetic cell, verify that **93 - 63 = 30** and the bridge residual
is zero. These exercises explain coverage and arithmetic; neither creates an estimate.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir(): ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from vehicle_tracker.cycles import read_cycle_history
from vehicle_tracker.events import vin_events, daily_counts
from vehicle_tracker.expectations import quarter_coverage, dated_input, revision_bridge, export_research


## Ordinary settings: explicit inputs, intentionally empty by default

Edit this cell to select retained evidence and analyst inputs. No company forecast
is filled in for you. Keep `CYCLE_REPORTS`, `ANALYST_INPUTS`, `FORECAST_PATHS` and
`REPORTED_RESULTS` empty for the teaching review; `BENCHMARK = None` means none selected.
The synthetic arithmetic later has its own fixed inputs and remains separate.


In [ ]:
AS_OF = '2026-09-08T13:00:00Z'
QUARTER = '2026Q3'
TIMEZONE = 'America/New_York'
DATABASE = ROOT / 'data/analysis/carvana_daily/history.sqlite'
CYCLE_REPORTS = []  # Existing cycle.json paths from one compatible inventory population.
ANALYST_INPUTS = []  # Explicit dated assumptions; field requirements appear below.
BENCHMARK = None
FORECAST_PATHS = []  # Existing frozen forecast JSON files; no save happens here.
REPORTED_RESULTS = []  # Explicit sourced quarterly result revisions, with both publication clocks.


### Advanced test/export inputs - normally unchanged

Automated checks and exports use the existing `*_OVERRIDE` hooks below. Ordinary
analyst choices belong in the preceding cell. Exporting/saving remains a separate
explicit action; normal Run All only reads retained files.


In [ ]:
AS_OF = globals().get('AS_OF_OVERRIDE', AS_OF)
QUARTER = globals().get('QUARTER_OVERRIDE', QUARTER)
DATABASE = Path(globals().get('DATABASE_OVERRIDE', DATABASE))
CYCLE_REPORTS = globals().get('CYCLE_REPORTS_OVERRIDE', CYCLE_REPORTS)
ANALYST_INPUTS = globals().get('ANALYST_INPUTS_OVERRIDE', ANALYST_INPUTS)
BENCHMARK = globals().get('BENCHMARK_OVERRIDE', BENCHMARK)
print('Review mode: fixed-cutoff quarterly replay | quarter:', QUARTER, '| cutoff:', AS_OF)
print('Selected cycle reports:', CYCLE_REPORTS)
print('Selected retained forecasts:', globals().get('FORECAST_PATHS_OVERRIDE', FORECAST_PATHS))


## 1. Read selected dates and coverage before estimates

`calendar` lists quarter dates; missing/partial dates remain unavailable. `research_status`
shows the selected population and freshness. Neither complete coverage nor a nonempty
activity table by itself supplies a calibrated national sales estimate.


In [ ]:
days, source_rows = read_cycle_history(CYCLE_REPORTS, DATABASE, as_of=AS_OF)
calendar, research_status = quarter_coverage(days, as_of=AS_OF, quarter=QUARTER, timezone_name=TIMEZONE)
display(research_status)
display(calendar)
print('Read-only database:', DATABASE, 'No collection, import or export occurs by default.')

In [ ]:
daily_activity = pd.DataFrame()
if not days.empty:
    events = vin_events(days, source_rows, absence_days=3)
    daily_activity = daily_counts(days, events)
    quarter_activity = calendar.merge(daily_activity[['cycle_date', 'observed_vins', 'pending_started', 'pending_cleared', 'persistent_absence']], on='cycle_date', how='left', validate='one_to_one')
    display(quarter_activity)
    quarter_activity.assign(date=pd.to_datetime(quarter_activity.cycle_date)).set_index('date')[['observed_vins', 'pending_started']].plot(subplots=True, figsize=(10, 5), title='Observed activity: missing dates stay gaps')
    plt.tight_layout()
    plt.show()
else:
    print('NO DAILY EVIDENCE. Historical intraday observations cannot fill missing quarter dates.')
print('Actual sales estimate:', research_status.estimated_retail_units.iloc[0])

## Optional analyst inputs: empty by default

Leave `ANALYST_INPUTS` empty and `BENCHMARK` unset for the first review. Their tables then say that no assumption or guidance/consensus comparison was selected. This is intentional.

An input needs an ID, source, availability timestamp, quarter, population (`scope_id`), metric, units and value. The reader checks the chosen version rather than silently selecting a newer one. Future, stale or mismatched inputs produce a **BLOCKED** explanation; they do not become zero.

The optional scenario uses this arithmetic:

`scenario units = persistent-absence events × assumed conversion + remaining quarter days × assumed daily units`

The activity is the persistent-absence event count from the chosen inventory population. Neither an absence nor the assumed conversion establishes a completed sale. The calculation requires complete selected quarter dates, observations no older than 36 hours and compatible inputs no older than seven days. Passing those checks would permit an exploratory scenario; it would not validate its assumptions or make the population nationally representative.

## Trace the synthetic teaching example

After the optional-input cell, start with `prior_daily` and `current_daily` in the teaching cell. Inspect their outer merge, then read `bridge` and the final one-row reconciliation. The few invented dates are compressed teaching inputs, not a complete quarter calendar.

- Previous scenario: activity `10 × 0.5` plus `29 × 2` remaining-day units = **63**.
- Current scenario: activity `15 × 0.6` plus `28 × 3` remaining-day units = **93**.
- Revision: **30 synthetic units**. The bridge separates revised old observations, new observations, one fewer remaining day, and the two changed assumptions.

Every number above is an assumption or synthetic activity count. None is a Carvana forecast.

In [ ]:
display(pd.DataFrame(ANALYST_INPUTS, columns=['input_id', 'source', 'available_at', 'quarter', 'scope_id', 'metric', 'units', 'value']))
benchmark_value = None
if BENCHMARK is not None:
    try:
        benchmark_value = dated_input(BENCHMARK, as_of=AS_OF, quarter=QUARTER,
            scope_id=research_status.scope_id.iloc[0], metric='retail_units', units='vehicles', max_age_days=7)
        print('Compatible benchmark:', benchmark_value, 'No spread calculated without a sales estimate.')
    except ValueError as error:
        print('BENCHMARK BLOCKED:', error)
else:
    print('No dated guidance/consensus selected. No benchmark or surprise is invented.')
analyst_scenario = pd.DataFrame()
if ANALYST_INPUTS:
    try:
        if not calendar.day_status.eq('complete').all() or research_status.age_hours.isna().any() or research_status.age_hours.iloc[0] > 36:
            raise ValueError('Complete fresh quarter dates are required; do not fill gaps with zero')
        inputs = pd.DataFrame(ANALYST_INPUTS).set_index('metric')
        if not inputs.index.is_unique: raise ValueError('Select one explicit version per input metric')
        conversion_record = inputs.loc['absence_conversion'].to_dict() | {'metric':'absence_conversion'}
        rate_record = inputs.loc['remaining_daily_units'].to_dict() | {'metric':'remaining_daily_units'}
        scope_id = research_status.scope_id.iloc[0]
        assumed_conversion = dated_input(conversion_record, as_of=AS_OF, quarter=QUARTER, scope_id=scope_id, metric='absence_conversion', units='units_per_absence', max_age_days=7)
        assumed_daily_rate = dated_input(rate_record, as_of=AS_OF, quarter=QUARTER, scope_id=scope_id, metric='remaining_daily_units', units='vehicles_per_day', max_age_days=7)
        if assumed_conversion > 1: raise ValueError('An absence conversion must be between zero and one')
        covered_activity = daily_activity[daily_activity.cycle_date.isin(calendar.cycle_date)].persistent_absence.sum(min_count=1)
        assumed_units_through_cutoff = covered_activity * assumed_conversion
        assumed_remaining_units = research_status.remaining_days.iloc[0] * assumed_daily_rate
        analyst_scenario = pd.DataFrame([dict(as_of=AS_OF, quarter=QUARTER, status='EXPLORATORY ASSUMPTION SCENARIO',
            activity=covered_activity, assumed_units_through_cutoff=assumed_units_through_cutoff,
            assumed_remaining_units=assumed_remaining_units, scenario_units=assumed_units_through_cutoff+assumed_remaining_units)])
        display(analyst_scenario)
        print('Scoped assumption scenario, not validated national sales. Actual sales estimate remains unavailable.')
    except (ValueError, KeyError) as error:
        print('ANALYST SCENARIO BLOCKED:', error)

In [ ]:
print('SYNTHETIC SCENARIO ONLY; not a calibrated Carvana estimate.')
example_source = dict(input_id='synthetic-conversion-v1', source='synthetic://teaching', available_at='2026-09-01T00:00:00Z', quarter='2026Q3', scope_id='synthetic', metric='conversion', units='units_per_activity', value=0.5)
conversion = dated_input(example_source, as_of='2026-09-01T23:59:00Z', quarter='2026Q3', scope_id='synthetic', metric='conversion', units='units_per_activity')
prior_daily = pd.DataFrame({'date':['2026-07-01', '2026-09-01'], 'activity':[6., 4.]})
current_daily = pd.DataFrame({'date':['2026-07-01', '2026-09-01', '2026-09-02'], 'activity':[6., 5., 4.]})
# Compressed teaching inputs, not complete observed quarter calendars.
comparison = prior_daily.merge(current_daily, on='date', how='outer', suffixes=('_previous', '_current'), indicator=True, validate='one_to_one')
display(comparison)
previous = dict(as_of='2026-09-01T23:59:00Z', quarter='2026Q3', scope_id='synthetic', metric='scenario_units', units='vehicles', coverage_basis='synthetic assumed complete quarter inputs', activity=float(prior_daily.activity.sum()), conversion=conversion, remaining_days=29, daily_rate=2.)
current = dict(previous, as_of='2026-09-02T23:59:00Z', activity=float(current_daily.activity.sum()), conversion=0.6, remaining_days=28, daily_rate=3.)
prior_scenario = previous['activity']*previous['conversion'] + previous['remaining_days']*previous['daily_rate']
current_scenario = current['activity']*current['conversion'] + current['remaining_days']*current['daily_rate']
revised_common_activity = float(comparison.loc[comparison['_merge'].eq('both'), 'activity_current'].sum())
bridge = revision_bridge(previous, current, prior_activity_revised=revised_common_activity)
display(bridge)
display(pd.DataFrame([dict(previous_scenario=prior_scenario, current_scenario=current_scenario, revision=current_scenario-prior_scenario, bridge_residual=current_scenario-prior_scenario-bridge.scenario_unit_change.sum())]))
try:
    revision_bridge(previous, dict(current, coverage_basis='partial coverage'), prior_activity_revised=revised_common_activity)
except ValueError as error:
    print('COVERAGE CHANGE BLOCKS ATTRIBUTION:', error)

## What would make a quarterly estimate supportable?

First inspect the bridge's `scenario_unit_change` column. Its five components sum to the displayed revision; `bridge_residual` should be zero. The order is explicit: revise common dates, add new dates, advance the calendar, change conversion, then change the remaining-day rate. Another order can assign interaction effects differently without changing the total.

A coverage/population mismatch raises **COVERAGE CHANGE BLOCKS ATTRIBUTION**. That teaching check prevents a changed sample from being described as sales acceleration.

For a real quarterly estimate, the missing evidence is comparable daily coverage across the stated population, later observations that test the proxy's rules, and a defensible mapping to Marketplace-inclusive retail units net of returns. Review immediate, three-day and seven-day absence rules in [Notebook 23](23_carvana_daily_sales_research.ipynb). The current narrow inventory history, native Sold labels and selected exit checks do not supply that mapping. A quarterly match cannot establish correct daily timing.

Keep four meanings separate: **inventory observation**, **native website status**, **assumption-based estimate**, and **independently confirmed transaction**. Pending clearance does not identify a cancellation; reappearance does not identify a returned purchase. Asking prices cannot supply realized revenue or earnings.

## Optional export: leave disabled for this review

The final cell leaves `EXPORT_DIRECTORY` unset. Ordinary Run All therefore writes no files. A separately requested export would require a new directory and would retain source/code hashes and assumptions; it would not turn the scenario into validated sales.

## Prospective forecast accuracy: target 1-3% quarterly absolute error

**Question:** how close were forecasts saved before the first public quarterly result to reported retail units?
**Inputs:** explicit frozen forecast JSON paths and sourced reported-result revisions. Both lists are empty by default. Website Sold labels and scoped assumption scenarios do not become company forecasts automatically.
**Calculation:** signed error = forecast units minus reported units; absolute percentage error (APE) = 100 times the absolute difference divided by reported units. A result below 1% is better than the target range. The <=3% column describes each vintage; it does not establish consistent accuracy.
**Interpretation:** positive signed error is overestimation. Inspect every forecast vintage and several later quarters, with a fixed forecast horizon and a designated evaluation vintage chosen before each result. Do not select the best forecast afterward or tune using the quarter you evaluate. Quarterly accuracy cannot validate daily sale dates.

`FORECAST_PATHS` in Ordinary settings selects retained forecast files. `REPORTED_RESULTS` supplies explicit dictionaries using the following fields. No result is fetched automatically. The corresponding `*_OVERRIDE` names remain available to tests/exports.

| Field | Meaning |
|---|---|
| `quarter`, `definition`, `reported_units` | Quarter, compatible metric and this revision's positive reported denominator. Use `carvana_retail_units_including_marketplace_net_returns`. |
| `available_at`, `source` | Time this particular result revision became public, and its source reference. Only revisions with `available_at <= AS_OF` may be evaluated. |
| `first_published_at`, `first_publication_source` | Verified first public release time for that quarter/definition and the evidence establishing that clock. For the original result, the two times can be equal. For a revision, retain the original first time. Missing/null first time leaves prospective eligibility unresolved. |

These are two different comparisons: **revision known** means `available_at <= AS_OF`; **prospective forecast** means `saved_at < first_published_at`. A forecast saved between the original release and a later revision remains ineligible, even when it happens to match that revision. Saving exactly at the first release is not strictly before it. Do not infer a first release from the earliest revision supplied; earlier releases may be absent from the input list. A publication date alone does not establish intraday availability.

The next cell first shows all supplied revisions known at the cutoff, then each forecast joined to the **latest available revision among those explicit inputs**. Supplying multiple revisions supports historical cutoff replay; it does not obtain omitted revisions. `known_result_revisions` counts the supplied revisions available at the cutoff. Different known first-publication clocks or duplicate revision timestamps fail validation. An earlier known revision can provide the common first-publication clock and source when a later revision omits them. A future revision cannot resolve an unknown first clock at an earlier cutoff. Older result inputs without first-publication metadata remain readable but unresolved. Changed frozen files and definition mismatches cannot establish prospective accuracy.

Saving remains a separate explicit operation via `vehicle/scripts/freeze_quarter_forecast.py`: preview an analyst JSON with `--input`, existing `--sources` (evidence plus model/assumptions), and a new `--destination`; add `--save` only to record it. Required fields are `forecast_id`, `quarter`, `forecast_units`, `as_of`, `model_version`, `model_description`, `trained_through_quarter`, `definition`, and `source_coverage`. Coverage includes `scope_id`, `description`, `expected_dates`, `complete_dates`, `partial_dates`, and `missing_dates`; its date counts must reconcile. The save retains source/model/input copies beside the JSON, then stamps the actual local clock and hashes. Editing a working model does not change a retained vintage. If copying is interrupted, preserve the partial folder and use a fresh destination; no final forecast JSON is published. Use distinct IDs for revisions and keep every miss. Local timestamps and hashes detect accidental changes; they are not independent proof against deliberate clock/file alteration.

No actual forecast is saved by this notebook. The 1-3% target remains unvalidated.


In [ ]:
from vehicle_tracker.expectations import forecast_review_rows
FORECAST_PATHS = globals().get('FORECAST_PATHS_OVERRIDE', globals().get('FORECAST_PATHS', []))
REPORTED_RESULTS = globals().get('REPORTED_RESULTS_OVERRIDE', globals().get('REPORTED_RESULTS', []))
forecast_vintages = forecast_review_rows(FORECAST_PATHS, REPORTED_RESULTS, as_of=AS_OF)
result_columns = ['quarter', 'definition', 'reported_units', 'available_at', 'source',
                  'first_published_at', 'first_publication_source']
reported_result_inputs = pd.DataFrame(REPORTED_RESULTS, columns=result_columns)
# Only a revision's own availability determines whether its contents are known.
known_reported_results = reported_result_inputs.loc[
    pd.to_datetime(reported_result_inputs.available_at, utc=True, format='ISO8601').le(pd.Timestamp(AS_OF))].copy()
display(known_reported_results)
print('Supplied future revisions excluded:', len(reported_result_inputs) - len(known_reported_results))
accuracy_rows = pd.DataFrame()
if forecast_vintages.empty:
    print('NO SAVED FORECASTS SELECTED. Prospective quarterly accuracy is unvalidated.')
else:
    display(forecast_vintages)  # Shows selected revision, both clocks, sources and unresolved vintages.
    # The helper validates and joins the latest supplied revision known at AS_OF.
    # Keep the strict first-publication comparison visible next to the error calculation.
    saved_before_first = pd.to_datetime(forecast_vintages.saved_at, utc=True, format='ISO8601').lt(
        pd.to_datetime(forecast_vintages.reported_first_published_at, utc=True, format='ISO8601'))
    accuracy_rows = forecast_vintages.loc[
        forecast_vintages.prospective_eligible.eq(True) & saved_before_first].copy()
    accuracy_rows['signed_error_units'] = accuracy_rows.forecast_units - accuracy_rows.reported_units
    accuracy_rows['signed_percentage_error'] = 100 * accuracy_rows.signed_error_units / accuracy_rows.reported_units
    accuracy_rows['absolute_percentage_error'] = 100 * accuracy_rows.signed_error_units.abs() / accuracy_rows.reported_units
    accuracy_rows['within_3_percent'] = accuracy_rows.absolute_percentage_error.le(3)
    display(accuracy_rows)
    print('Distinct evaluated quarters:', accuracy_rows.quarter.nunique(),
          '| Forecast vintages:', len(accuracy_rows), '| Target: 1-3% quarterly APE.')
    print('Inspect bias and all vintages. Several later quarters at a fixed horizon are needed; daily timing remains unvalidated.')


In [ ]:
# These tables and input paths also support the separate offline export CLI.
research_tables = {'quarter_coverage': calendar, 'research_status': research_status,
    'observed_activity': daily_activity, 'analyst_scenario': analyst_scenario,
    'known_reported_results': known_reported_results, 'forecast_vintages': forecast_vintages,
    'accuracy_rows': accuracy_rows,
    'analyst_inputs': pd.DataFrame(ANALYST_INPUTS),
    'benchmark_input': pd.DataFrame([BENCHMARK]) if BENCHMARK is not None else pd.DataFrame()}
input_paths = [ROOT / 'notebooks/30_carvana_sales_expectations.ipynb']
input_paths += [Path(path) for path in CYCLE_REPORTS]
input_paths += [Path(path) for path in FORECAST_PATHS]
if CYCLE_REPORTS and DATABASE.is_file(): input_paths.append(DATABASE)
from vehicle_tracker.cycles import cycle_evidence
for path in CYCLE_REPORTS:
    _, _, reports = cycle_evidence(path, as_of=AS_OF)
    input_paths += reports
    for report in reports:
        input_paths += [Path(page['retained_source']) for page in json.loads(report.read_text())['pages']]
if not forecast_vintages.empty:
    for path in forecast_vintages.forecast_path:
        frozen = json.loads(Path(path).read_text(encoding='utf-8'))
        input_paths += [Path(path) for path in frozen['source_sha256']]
input_paths = list(dict.fromkeys(input_paths))
EXPORT_DIRECTORY = globals().get('EXPORT_DIRECTORY_OVERRIDE')  # Explicit NEW directory only.
if EXPORT_DIRECTORY is not None:
    export_research(research_tables, destination=EXPORT_DIRECTORY, as_of=AS_OF, source_paths=input_paths,
        assumptions={'quarter':QUARTER, 'timezone':TIMEZONE, 'absence_days':3, 'max_age_hours':36,
            'analyst_inputs':ANALYST_INPUTS, 'benchmark':BENCHMARK,
            'reported_result_selection':'Latest supplied revision available at cutoff; eligibility uses sourced first publication'})
else:
    print('Export disabled. A separate explicit export records source/code/notebook hashes, result vintages and assumptions.')
